# Распределение товаров по торговым точкам через роевой интеллект

## Библиотеки

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

## Параметры

Корневая папка со всеми датасетами

In [2]:
root_dir = Path(r"./datasets")

Закрывающийся магазин. Указать id в виде строки

In [3]:
closing_store = "10"

Считать ли алкогольные товары отдельно. Это нужно, так как не у всех организаций есть лицензия на продажу алкоголя. В таком случае, нужно искать наилучшее распределение среди магазинов, которые состоят в той же организации, что и закрывающийся

In [4]:
separate_alcohol = True

Минимальный срок годности распределяемого товара. Это обусловлено тем, что процесс закрытия магазина не быстрый, обычно занимает не меньше месяца времени, потому скоропортящиеся товары в любом случае не успеют быть распределены.

Значение переменной 0 означает, что срок годности любой

In [5]:
spoilage_threshold = 30

Может быть такое, что магазины уже перегружены распределяемым товаром. Это переменная определяет, что делать в таком случае. Если значение True - то тогда магазин всё ещё считается в распределении, просто будет штраф к оценке. Если значение False - то тогда магазин исключается из распределения

In [6]:
ignore_overload = False

Стоит ли искать магазины с тем же куратором или организацией, что и закрывающийся магазин. Это нужно, так как контакты между разными организациями или кураторами добавляют лишние издержки, проще когда всё происходит в рамках одной компании или куратора.

Важно отметить, что если `separate_alcohol` истинно, то и `same_organization` должно быть истинно, так как не все организации могут торговать алкоголем

In [7]:
same_curator = False
same_organization = False

## Эмпирики

## Реализация

In [8]:
assortment = pd.read_csv(root_dir / "Ассортимент" / "Assortment.csv", sep=";", dtype={"№Магазина" : "str"})  # т.к. под номером может быть РЦ
assortment.head()

,№Магазина,Номер SKU,Ассортимент Статус,Ассортимент Вид,КоличествоКонечныйОстатокНП
0,54,108283,Введен,Регулярный,"-0,510"
1,54,223588,Введен,Регулярный,"-1,422"
2,54,92602,Введен,Регулярный,"-1,504"
3,54,188801,Введен,Регулярный,"3,304"
4,54,169467,Введен,Регулярный,"-6,256"


In [9]:
leftovers = pd.read_csv(root_dir / "Остатки" / "Stocks.csv", sep=";", dtype={"номер магазина" : "str"})
leftovers.head()

,номер магазина,код товара,остаток,остаток_робот
0,05,215673,"4,000","4,000"
1,75,215673,"8,000","8,000"
2,137,226249,"24,000","24,000"
3,39,226249,"29,000","29,000"
4,08,226249,"30,000","30,000"


In [10]:
facings = pd.read_csv(root_dir / "Планограмма" / "Export_facings.csv", sep=";", dtype={"STORENUMBER" : "str"})
facings.head()

,STORENUMBER,SECTIONID,STOCKCODE,XFACINGS,YFACINGS,ZFACINGS,MAX_CAPACITY,MIN_CAPACITY
0,10,20055,13880,3,1,5,15,6
1,10,20055,13881,3,1,5,15,6
2,10,20055,15794,1,1,7,7,2
3,10,20055,36571,2,1,6,12,4
4,10,20055,36571,2,1,6,12,4


In [11]:
stores = pd.read_excel(root_dir / "SPR_Магазины_Сеть_РД.xlsx", dtype={"Магазин" : "str"})
stores.head()

,Магазин,Сеть,Типоразмер,Куратор,Организация,Дата открытия,"Площадь общая, м2","Площадь ТЗ, м2",Район,Географическое положение,Формат,Ассортимент,Тип цен реализации,Режим работы,Телефон,Адрес,Номер группы
0,2,Сеть 1,NaN,Куратор 3,Фирма 4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3,Сеть 1,NaN,Куратор 3,Фирма 4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4,Сеть 1,NaN,Куратор 3,Фирма 4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,5,Сеть 1,NaN,Куратор 2,Фирма 1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,6,Сеть 1,NaN,Куратор 4,Фирма 1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
nomenclature = pd.read_excel(root_dir / "SPR_Номенклатура_NEW.xlsx")
nomenclature.head()

,Номенклатура,Срок хранения товара,Код,Штрих-Код,Уровень2,Уровень3,Уровень4,Уровень5,KVI,ТПЦ,...,Номенклатура.Производитель,Номенклатура.Поставщик,Номенклатура.Бренд,Номенклатура.СТМ,Номенклатура.Эксклюзив,Номенклатура.Базовая единица измерения,Номенклатура.Поставщик.Код,Возвратный товар,СреднийВес,ОсновнаяЕдиницаИзмерения
0,Товар 1,0,1,NaN,04 Замороженная продукция,0403 Рыба и морепродукты замороженные,040301 Рыба и Филе рыбы зам,04030102 Рыба белая тушка зам,Нет,Нет,...,NaN,Тер-Тумасов В.Б. ИП,NaN,Нет,Нет,кг,ЦБ000464,NaN,NaN,кг
1,Товар 2,0,756,NaN,07 Непродовольственные товары,0705 Товары для уборки,"070501 Губки, тряпки, швабры",07050103 Губки для посуды,Нет,Нет,...,NaN,Бененсон А.Л. ИП,NaN,Нет,Нет,шт,ЦБ000463,NaN,NaN,шт
2,Товар 3,0,766,NaN,07 Непродовольственные товары,"0714 Товары для праздника, Сувениры",071405 Новогодний ассортимент,"07140503 Гирлянды, мишура",Нет,Нет,...,NaN,Бененсон А.Л. ИП,Vegas,Нет,Нет,шт,ЦБ000463,NaN,NaN,шт
3,Товар 4,0,773,NaN,07 Непродовольственные товары,"0714 Товары для праздника, Сувениры",071405 Новогодний ассортимент,"07140503 Гирлянды, мишура",Нет,Нет,...,NaN,Бененсон А.Л. ИП,Vegas,Нет,Нет,шт,ЦБ000463,NaN,NaN,шт
4,Товар 5,0,812,NaN,07 Непродовольственные товары,"0714 Товары для праздника, Сувениры",071405 Новогодний ассортимент,"07140503 Гирлянды, мишура",Нет,Нет,...,NaN,Бененсон А.Л. ИП,Vegas,Нет,Нет,шт,ЦБ000463,NaN,NaN,шт


In [13]:
sales = pd.read_csv(root_dir / "Продажи" / "SPR_Среднедневные_продажи.csv", sep=",", dtype={"Магазин" : "str"})
sales.head()

,Магазин,Код товара,"Продажи, ед. изм."
0,2,756,1.191209
1,2,773,0.972093
2,2,906,1.145055
3,2,909,0.681319
4,2,945,0.426637


Переименование одинаковых колонок

In [14]:
col_vol = "Объём товара в закрывающемся магазине (V)"
col_maxcap = "Максимальная вместимость товара в магазине (M)"
col_item = "Код товара"
col_store = "Магазин"
col_status = "Ассортимент Статус"
col_verdict = "Итоговый вердикт"
col_rating = "Итоговая оценка"
col_remain = "Остаток товара в магазине (R)"
col_space = "Свободные места для товара (M-R)"
col_load = "Процент загруженности (R/M)"
col_sales = "Продажи товара в магазине (S)"
col_eff = "Эффективность продажи ((V+R)/S)"
col_coef = "Итоговый коэффициент"
col_coef_eff = "Коэффициент эффективности продаж"
col_coef_load = "Коэффициент загруженности"
col_best = "Наилучший магазин по коэффициенту"

In [15]:
assortment = assortment.rename(columns={"№Магазина" : col_store, "Номер SKU" : col_item})
leftovers = leftovers.rename(columns={"номер магазина" : col_store, "код товара" : col_item})
facings = facings.rename(columns={"STORENUMBER" : col_store, "STOCKCODE" : col_item})
stores = stores.rename(columns={"Магазин" : col_store})
nomenclature = nomenclature.rename(columns={"Код" : col_item})
sales = sales.rename(columns={"Код товара" : col_item, "Магазин" : col_store, "Продажи, ед. изм." : col_sales})

In [16]:
leftovers["остаток_робот"] = leftovers["остаток_робот"].replace(r"\s+", "", regex=True)  # убрать пробелы в числах
leftovers["остаток_робот"] = leftovers["остаток_робот"].replace(",", ".", regex=True)  # т.к. по стандарту разделение через .
leftovers["остаток_робот"] = pd.to_numeric(leftovers["остаток_робот"])
leftovers["остаток_робот"] = leftovers["остаток_робот"].fillna(0)
leftovers.head()

,Магазин,Код товара,остаток,остаток_робот
0,05,215673,"4,000",4.0
1,75,215673,"8,000",8.0
2,137,226249,"24,000",24.0
3,39,226249,"29,000",29.0
4,08,226249,"30,000",30.0
